In [3]:
from openai import OpenAI
import json
import requests

In [4]:
model="gpt-4o-mini"
client=OpenAI(
    api_key=""
)

In [ ]:
def get_weather(latitude, longitude):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data['current']['temperature_2m']

tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number", "description": "Latitude of the location."},
            "longitude": {"type": "number", "description": "Longitude of the location."}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

input_message=[{"role":"user","content":"what s the weather like in paris today?"}]

response=client.responses.create(
    model=model,
    input=input_message,
    tools=tools,
)


print("get weather response output:",response.output)

if  response.output  and response.output[0]=="function":
    tool_call=response.output[0]
    args=json.loads(tool_call["latitude"]["longitude"])
    weahther_result=get_weather(args["latitude"], args["longitude"])
    print(f"weather result: {weahther_result}C")

    input_message.append(tool_call)
    input_message.append({
        "type":"function_call_output",
    "call_id":tool_call.call_id,
        "output":str(json.dumps(weahther_result)
                     )})

response=client.responses.create(
    model=model,
    input=input_message,
    tools=tools,
)
print("get weather response output:",response.output_text)

In [ ]:
tools = [{
    "type": "web_search_preview"
}]

input_message = "What was a positive news story from today?"

response = client.responses.create(
    model=model,
    tools=tools,
    input=[{"role": "user", "content": input_message}]
)

print("Web Search Tool Response Output:", response.output_text)


tools_with_location = [{
    "type": "web_search_preview",
    "user_location": {
         "type": "approximate",
         "country": "GB",
         "city": "London",
         "region": "London"
    },
    "search_context_size": "low"
}]

response = client.responses.create(
    model=model,
    tools=tools_with_location,
    input=[{"role": "user", "content": "What are the best restaurants around Granary Square?"}]
)

print("Web Search with Location Response Output:", response.output_text)

In [ ]:
tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for a given location.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City and country e.g. Bogotá, Colombia"
            }
        },
        "required": [
            "location"
        ],
        "additionalProperties": False
    }
}]

stream = client.responses.create(
    model=model,
    input=[{"role": "user", "content": "What's the weather like in Paris today?"}],
    tools=tools,
    stream=True
)

for event in stream:
    print(event)